# 🧠 Notebook 15: Performance and Benchmarking

## 1. Purpose + Scope

This notebook covers how T81 measures performance:

*   **Deterministic Benchmarking Methodology**: Measuring "work units" (gas) instead of wall-clock time.
*   **Memory Behavior Exploration**: Allocations and cache efficiency.
*   **SIMD Acceleration Explanation**: How AVX2/NEON are used safely.
*   **Reproducibility Commentary**: Why benchmarks must be stable.

## 2. Spec References

*   `benchmarks/` directory
*   `include/t81/simd/simd.hpp`

## 3. Determinism Tier

**Tier A (Strict Determinism)**: Benchmarks of logical work (instructions executed) are deterministic. Wall-clock benchmarks are informative but platform-dependent.

## 4. Reproducibility Setup

Ensure `t81_python` is built and available in `PYTHONPATH`.

In [ ]:
import sys
import os

build_dir = os.path.abspath(os.path.join(os.getcwd(), "../build"))
if build_dir not in sys.path:
    sys.path.append(build_dir)

try:
    import t81_python
    print("✅ t81_python loaded.")
except ImportError:
    print("❌ Failed to load t81_python.")
    sys.exit(1)

## 5. Logical Work Measurement

We measure steps, not seconds.

In [ ]:
src = "fn work() -> T81BigInt { return 1t81 + 1t81; }"
try:
    prog = t81_python.compile(src)
    vm = t81_python.make_interpreter_vm()
    vm.load_program(prog)
    vm.run_to_halt()
    trace = vm.trace
    print(f"Instructions executed: {len(trace)}")
except RuntimeError as e:
    print(f"❌ Benchmarking failed: {e}")
# This number will be the same on every machine.

## 6. SIMD Acceleration

SIMD instructions process multiple data points at once. T81 uses them for BigInt and Tensor ops, but ensures the result is bit-exact to the scalar implementation.

In [ ]:
# Conceptual verification
print("SIMD results are verified against scalar reference implementations in unit tests (e.g., t81_base81_simd_test).")

## 7. Memory Behavior

Monitoring heap usage is critical.

## 8. Architectural Commentary

Performance is important, but correctness and determinism are paramount. T81 optimizes heavily (SIMD, allocation arenas) but never at the expense of consensus safety.